# TACTIC-Review: ESP-IDF Colab Feasibility Pilot

This notebook reproduces the clean baseline and one controlled target-specific
build task for ESP32 and ESP32-S3 using ESP-IDF v5.5.4.

Run all cells from top to bottom. No physical ESP32 board is required.


In [ ]:
import os, platform, shutil
print("Python:", platform.python_version())
print("System:", platform.platform())
total, used, free = shutil.disk_usage("/content")
print("Runtime disk total (GB):", round(total / 1024**3, 2))
print("Runtime disk free (GB):", round(free / 1024**3, 2))


In [ ]:
import os, subprocess, time
IDF_VERSION = "v5.5.4"
IDF_PATH = "/content/esp-idf"

if os.path.exists(IDF_PATH):
    subprocess.run(["rm", "-rf", IDF_PATH], check=True)

start = time.time()
result = subprocess.run(
    ["git", "clone", "--branch", IDF_VERSION, "--depth", "1",
     "--recursive", "--shallow-submodules",
     "https://github.com/espressif/esp-idf.git", IDF_PATH],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print(result.stdout[-4000:])
if result.returncode != 0:
    raise RuntimeError("ESP-IDF download failed")
print("Elapsed minutes:", round((time.time() - start) / 60, 2))


In [ ]:
import subprocess, os, shutil, time

subprocess.run(
    ["bash", "-lc",
     "apt-get update -qq && apt-get install -y python3.12-venv python3-venv python3-pip ninja-build"],
    check=True
)

py_env = "/root/.espressif/python_env/idf5.5_py3.12_env"
if os.path.exists(py_env):
    shutil.rmtree(py_env)

result = subprocess.run(
    ["bash", "-lc", f"cd {IDF_PATH} && ./install.sh esp32,esp32s3"],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print(result.stdout[-8000:])
if result.returncode != 0:
    raise RuntimeError("ESP-IDF tool installation failed")


In [ ]:
import subprocess
result = subprocess.run(
    ["bash", "-lc", f'''
    set -e
    source {IDF_PATH}/export.sh
    idf.py --version
    xtensa-esp-elf-gcc --version | head -n 1
    cmake --version | head -n 1
    ninja --version
    python --version
    '''],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print(result.stdout)
if result.returncode != 0:
    raise RuntimeError("Environment verification failed")


In [ ]:
import os, shutil, subprocess, time, json

SOURCE_PROJECT = f"{IDF_PATH}/examples/get-started/hello_world"
WORK_ROOT = "/content/tactic_pilot"
RESULTS_DIR = f"{WORK_ROOT}/results"

shutil.rmtree(WORK_ROOT, ignore_errors=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

projects = {
    "esp32": f"{WORK_ROOT}/hello_esp32",
    "esp32s3": f"{WORK_ROOT}/hello_esp32s3"
}
for p in projects.values():
    shutil.copytree(SOURCE_PROJECT, p)

def build_target(target, project_path, prefix):
    log_path = f"{RESULTS_DIR}/{prefix}_{target}.log"
    cmd = f'''
    set -o pipefail
    source {IDF_PATH}/export.sh >/dev/null 2>&1
    cd {project_path}
    idf.py set-target {target}
    idf.py build
    '''
    start = time.time()
    result = subprocess.run(["bash", "-lc", cmd], text=True,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    elapsed = round(time.time() - start, 2)
    open(log_path, "w", encoding="utf-8").write(result.stdout)
    return {
        "target": target,
        "status": "PASS" if result.returncode == 0 else "FAIL",
        "exit_code": result.returncode,
        "elapsed_seconds": elapsed,
        "log_file": log_path
    }

baseline_results = [build_target(t, p, "baseline") for t, p in projects.items()]
baseline = {
    "esp_idf_version": "v5.5.4",
    "sample_project": "examples/get-started/hello_world",
    "evaluation_stage": "clean baseline",
    "results": baseline_results
}
open(f"{RESULTS_DIR}/baseline_summary.json", "w").write(json.dumps(baseline, indent=2))
print(json.dumps(baseline, indent=2))


In [ ]:
from pathlib import Path
import os, shutil, subprocess, time, json

TASK_ROOT = f"{WORK_ROOT}/task_01_corrected"
shutil.rmtree(TASK_ROOT, ignore_errors=True)

task_projects = {
    "esp32": f"{TASK_ROOT}/changed_esp32",
    "esp32s3": f"{TASK_ROOT}/changed_esp32s3"
}
for p in task_projects.values():
    shutil.copytree(SOURCE_PROJECT, p)

CONTROLLED_DEFECT = r'''
/*
 * TACTIC Task 01: controlled target-specific build defect.
 * Expected:
 *   ESP32    -> FAIL
 *   ESP32-S3 -> PASS
 */
#include "sdkconfig.h"

#if defined(CONFIG_IDF_TARGET_ESP32) && CONFIG_IDF_TARGET_ESP32
#error "TACTIC_TASK_01: controlled ESP32-only build failure"
#endif
'''

def find_app_main(project_path):
    for candidate in Path(project_path).rglob("*.c"):
        if "app_main" in candidate.read_text(encoding="utf-8", errors="ignore"):
            return candidate
    raise FileNotFoundError("app_main source not found")

for target, project_path in task_projects.items():
    source_file = find_app_main(project_path)
    original = source_file.read_text(encoding="utf-8")
    source_file.write_text(CONTROLLED_DEFECT + "\n" + original, encoding="utf-8")

def configure(target, project_path):
    result = subprocess.run(
        ["bash", "-lc", f'''
        set -o pipefail
        source {IDF_PATH}/export.sh >/dev/null 2>&1
        cd {project_path}
        idf.py set-target {target}
        grep '^CONFIG_IDF_TARGET=' sdkconfig || true
        grep '^CONFIG_IDF_TARGET_ESP32=' sdkconfig || true
        grep '^CONFIG_IDF_TARGET_ESP32S3=' sdkconfig || true
        '''],
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    print(result.stdout)
    if result.returncode != 0:
        raise RuntimeError(f"Configuration failed for {target}")

for target, project_path in task_projects.items():
    configure(target, project_path)

def build_changed(target, project_path):
    log_path = f"{RESULTS_DIR}/task_01_corrected_changed_{target}.log"
    start = time.time()
    result = subprocess.run(
        ["bash", "-lc", f'''
        set -o pipefail
        source {IDF_PATH}/export.sh >/dev/null 2>&1
        cd {project_path}
        idf.py build
        '''],
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    elapsed = round(time.time() - start, 2)
    open(log_path, "w", encoding="utf-8").write(result.stdout)
    marker = "TACTIC_TASK_01: controlled ESP32-only build failure"
    return {
        "target": target,
        "status": "PASS" if result.returncode == 0 else "FAIL",
        "exit_code": result.returncode,
        "elapsed_seconds": elapsed,
        "expected_status": "FAIL" if target == "esp32" else "PASS",
        "failure_marker_found": marker in result.stdout,
        "log_file": log_path
    }

esp32 = build_changed("esp32", task_projects["esp32"])
esp32s3 = build_changed("esp32s3", task_projects["esp32s3"])

supported = (
    esp32["status"] == "FAIL"
    and esp32["failure_marker_found"]
    and esp32s3["status"] == "PASS"
)

evidence = {
    "task_id": "TACTIC_TASK_01_CORRECTED",
    "task_type": "controlled target-specific build defect",
    "claim": "The injected change breaks the ESP32 build while preserving the ESP32-S3 build.",
    "environment": {
        "esp_idf_version": "v5.5.4",
        "sample_project": "examples/get-started/hello_world",
        "targets": ["esp32", "esp32s3"]
    },
    "baseline": {"esp32": "PASS", "esp32s3": "PASS"},
    "changed_results": {"esp32": esp32, "esp32s3": esp32s3},
    "admission_decision": "ADMIT" if supported else "ABSTAIN",
    "claim_supported": supported
}

open(f"{RESULTS_DIR}/task_01_corrected_evidence.json", "w").write(
    json.dumps(evidence, indent=2)
)
print(json.dumps(evidence, indent=2))


In [ ]:
# Optional: package the generated logs and JSON files for download.
import shutil
archive = shutil.make_archive("/content/TACTIC_Pilot_Runtime_Results", "zip",
                              "/content/tactic_pilot/results")
print(archive)
